In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp "/content/drive/Shareddrives/Hotels/Full_HotelRec/HotelRec.db" "/content/hotelrec.db" # copy db file from drive storage to local colab storage for faster processing later

In [ ]:
import sqlite3
import pandas as pd

#db_path = '/content/drive/Shareddrives/Hotels/Full_HotelRec/HotelRec.db'
db_path = '/content/hotelrec.db'


# Connect to the SQLite database
conn = sqlite3.connect(db_path)
print(f"Successfully connected to {db_path}")

# Let's see what tables are available in the database
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("\nTables in the database:")
for table in tables:
    print(table[0])

Successfully connected to /content/hotelrec.db

Tables in the database:
hotel_reviews


In [ ]:
import xgboost as xgb
print(xgb.__version__)

3.2.0


In [ ]:
# Show schema
schema_df = pd.read_sql_query("PRAGMA table_info(hotel_reviews);", conn)
print(schema_df)

# Show a few rows
sample_df = pd.read_sql_query("SELECT * FROM hotel_reviews LIMIT 5;", conn)
sample_df

In [ ]:
quick_years = pd.read_sql_query("""
SELECT SUBSTR(date, 1, 4) AS year, COUNT(*) AS n
FROM (
    SELECT date
    FROM hotel_reviews
    LIMIT 5000000
)
GROUP BY year
ORDER BY year;
""", conn)

print(quick_years)

# view distribution of years from sample of data

In [ ]:
eligible_users = pd.read_sql_query("""
    SELECT DISTINCT author
    FROM hotel_reviews
    WHERE date >= '2016-01-01'
""", conn)

sampled_users = eligible_users.sample(n=100000, random_state=42)

conn.execute("DROP TABLE IF EXISTS sampled_users;")
conn.execute("CREATE TEMP TABLE sampled_users(author TEXT);")

conn.executemany(
    "INSERT INTO sampled_users(author) VALUES (?);",
    [(u,) for u in sampled_users["author"].tolist()]
)
conn.commit()

subset_df = pd.read_sql_query("""
    SELECT *
    FROM hotel_reviews
    WHERE date >= '2016-01-01'
      AND author IN (SELECT author FROM sampled_users);
""", conn)

print("rows:", len(subset_df))
print("users:", subset_df["author"].nunique())
print("hotels:", subset_df["hotel_url"].nunique())
print("avg reviews/user:", subset_df.groupby("author").size().mean())
print("median reviews/user:", subset_df.groupby("author").size().median())
print("avg reviews/hotel:", subset_df.groupby("hotel_url").size().mean())
print("median reviews/hotel:", subset_df.groupby("hotel_url").size().median())

# Applied temporal filter, cutoff taking 2016 onwards
# Chose temporal to improve data relevance and maintain density
# From this filtered subset, randomly sampled users, and kept all interactions (rows) from these users witin 2016-onwards window
# Chose user sampling instead of row sampling to keep full behavior for eaach user intact

In [ ]:
cols_to_drop = ["text", "title"]
subset_df = subset_df.drop(columns=cols_to_drop)
subset_df.to_csv("hotelrec_subset_2016plus_100kusers.csv", index=False)

# drop unused title and text and save subset csv

In [ ]:
df = pd.read_csv("hotelrec_subset_2016plus_100kusers.csv")

print(df.shape)
print(df.columns)
print(df.dtypes)

# sanity check, examine subset csv

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# convert date to datetime and sort

In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

# add basic time features

In [ ]:
# User prior review count
df["user_review_count_prior"] = df.groupby("author").cumcount()

# User prior average rating
df["user_avg_rating_prior"] = (
    df.groupby("author")["rating"]
      .apply(lambda x: x.expanding().mean().shift(1))
      .reset_index(level=0, drop=True)
)

#for each user find how many past reviews they have written before the current one and the average of their prior ratings

In [ ]:
# Hotel prior review count
df["hotel_review_count_prior"] = df.groupby("hotel_url").cumcount()

# Hotel prior average rating
df["hotel_avg_rating_prior"] = (
    df.groupby("hotel_url")["rating"]
      .apply(lambda x: x.expanding().mean().shift(1))
      .reset_index(level=0, drop=True)
)

# for each hotel find how many past reviews it has before the current one and the average of its prior ratings

In [ ]:
df[[
    "author", "hotel_url", "date", "rating",
    "user_review_count_prior", "user_avg_rating_prior",
    "hotel_review_count_prior", "hotel_avg_rating_prior"
]].head(15)

# sanity check to ensure no leakage (prevent model from seeing future info which give artifically high performance)

In [ ]:
df["user_has_history"] = (df["user_review_count_prior"] > 0).astype(int)
df["hotel_has_history"] = (df["hotel_review_count_prior"] > 0).astype(int)

# add binary indicators to explicitly inform xgboost of cold start cases (no prior interactions)

In [ ]:
# add aspect based features for hotels
aspects = ["sleep_quality", "value", "rooms", "service", "cleanliness", "location"]

for col in aspects:
    df[f"hotel_avg_{col}_prior"] = (
        df.groupby("hotel_url")[col]
          .apply(lambda x: x.expanding().mean().shift(1))
          .reset_index(level=0, drop=True)
    )

# for each hotel and each aspect, find average service, cleanliness, etc before this review
# opt to not use user based features because user history is weaker, can decide to add later if needed

In [ ]:
df[[
    "hotel_url", "date",
    "service",
    "hotel_avg_service_prior"
]].head(15)

# sanity check, first occurrence should be nan and later averages build correctly

In [ ]:
# 90/5/5 train/validate/test split, preserving temporal ordering
n = len(df)

train_end = int(0.90 * n)
val_end   = int(0.95 * n)

train = df.iloc[:train_end]
val   = df.iloc[train_end:val_end]
test  = df.iloc[val_end:]
# chose 90/5/5 to maximize data available for training given this dataset's extreme sparsity

# check that train < val < test chronologically
print(train["date"].min(), train["date"].max())
print(val["date"].min(), val["date"].max())
print(test["date"].min(), test["date"].max())

In [ ]:
# build feature matrix to use for xgboost training
features = [
    col for col in df.columns
    if col not in ["author", "hotel_url", "date", "rating"]
]
#drop non-feature cols, to model: (user behavior, hotel behavior, context) → rating

In [ ]:
#create datasets
X_train = train[features]
y_train = train["rating"]

X_val = val[features]
y_val = val["rating"]

X_test = test[features]
y_test = test["rating"]

In [ ]:
# train xgboost w/ 200 estimators
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

In [ ]:
# evaluate w/200 estimators
from sklearn.metrics import mean_squared_error
import numpy as np

preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print("Test RMSE:", rmse)

In [ ]:
# print feature importance for report
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X_train.columns)
print(importance.sort_values(ascending=False).head(15))

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
from itertools import product

param_grid = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5]
}

results = []

keys = list(param_grid.keys())
values = list(param_grid.values())

for combo in product(*values):
    params = dict(zip(keys, combo))

    model = XGBRegressor(
        n_estimators=1000,
        early_stopping_rounds=50,
        random_state=42,
        **params
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    preds_val = model.predict(X_val)
    rmse_val = np.sqrt(mean_squared_error(y_val, preds_val))

    results.append({
        **params,
        "best_iteration": model.best_iteration,
        "val_rmse": rmse_val
    })

results_df = pd.DataFrame(results).sort_values("val_rmse").reset_index(drop=True)
print(results_df.head(10))

In [ ]:
# train final model using best row from results_df
best_params = results_df.iloc[0].to_dict() #cast parameters to int to fix typing error
best_params.pop("val_rmse")
best_params.pop("best_iteration")

best_params["max_depth"] = int(best_params["max_depth"])
best_params["min_child_weight"] = int(best_params["min_child_weight"])
best_params["learning_rate"] = float(best_params["learning_rate"])
best_params["subsample"] = float(best_params["subsample"])
best_params["colsample_bytree"] = float(best_params["colsample_bytree"])

final_model = XGBRegressor(
    n_estimators=1000,
    early_stopping_rounds=50,
    random_state=42,
    **best_params
)

final_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

preds_test = final_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, preds_test))

print("Best params:", best_params)
print("Test RMSE:", test_rmse)

===========================

XGBOOST WITH FULL DATASET

===========================

In [ ]:
# run first three cells of notebook before this

pd.read_sql_query("PRAGMA index_list('hotel_reviews');", conn) # check if index on date exists

# date index does not exist

,seq,name,unique,origin,partial
0,0,idx_date,0,c,0


In [ ]:
conn.execute("CREATE INDEX IF NOT EXISTS idx_date ON hotel_reviews(date);")
conn.commit()

In [ ]:
import time

query = """
SELECT hotel_url, author, date, rating,
       CAST(substr(date, 1, 4) AS INTEGER) AS year,
       CAST(substr(date, 6, 2) AS INTEGER) AS month,
       sleep_quality, value, rooms, service, cleanliness, location
FROM hotel_reviews
ORDER BY date
"""

chunk_iter = pd.read_sql_query(query, conn, chunksize=50_000)

start_time = time.time()
total_rows_processed = 0

In [ ]:
from collections import defaultdict
import pandas as pd

# running stats for overall ratings
user_rating_sum = defaultdict(float)
user_rating_count = defaultdict(int)

hotel_rating_sum = defaultdict(float)
hotel_rating_count = defaultdict(int)

# running stats for hotel aspect averages
aspects = ["sleep_quality", "value", "rooms", "service", "cleanliness", "location"]
hotel_aspect_sum = {a: defaultdict(float) for a in aspects}
hotel_aspect_count = {a: defaultdict(int) for a in aspects}

#output_path = "hotelrec_model_ready_full_streamed.csv"
# output_path = "/content/drive/MyDrive/hotelrec_model_ready_full_streamed.csv"
output_path = "/content/hotelrec_model_ready_full_streamed.csv"
first_write = True

for chunk_num, chunk in enumerate(chunk_iter, start=1):
    chunk_start = time.time()
    chunk_size = len(chunk)
    total_rows_processed += chunk_size

    # print(f"\n--- Processing chunk {chunk_num} ---")
    # print(f"Chunk size: {chunk_size:,}")
    # print(f"Total rows processed so far: {total_rows_processed:,}")

    print(f"\n--- Processing chunk {chunk_num} ---", flush=True)
    print(f"Chunk size: {chunk_size:,}", flush=True)
    print(f"Total rows processed so far: {total_rows_processed:,}", flush=True)

    rows_out = []

    for row in chunk.itertuples(index=False):
        user = row.author
        hotel = row.hotel_url
        rating = row.rating

        # prior user features
        user_count_prior = user_rating_count[user]
        user_avg_rating_prior = (
            user_rating_sum[user] / user_count_prior
            if user_count_prior > 0 else None
        )

        # prior hotel features
        hotel_count_prior = hotel_rating_count[hotel]
        hotel_avg_rating_prior = (
            hotel_rating_sum[hotel] / hotel_count_prior
            if hotel_count_prior > 0 else None
        )

        # build output row
        out = {
            "date": row.date,
            "rating": rating,
            "year": row.year,
            "month": row.month,
            "user_review_count_prior": user_count_prior,
            "user_avg_rating_prior": user_avg_rating_prior,
            "hotel_review_count_prior": hotel_count_prior,
            "hotel_avg_rating_prior": hotel_avg_rating_prior,
            "user_has_history": int(user_count_prior > 0),
            "hotel_has_history": int(hotel_count_prior > 0),
        }

        # prior hotel aspect averages
        for a in aspects:
            cnt = hotel_aspect_count[a][hotel]
            out[f"hotel_avg_{a}_prior"] = (
                hotel_aspect_sum[a][hotel] / cnt if cnt > 0 else None
            )

        rows_out.append(out)

        # update AFTER computing features
        if pd.notna(rating):
            user_rating_sum[user] += rating
            user_rating_count[user] += 1

            hotel_rating_sum[hotel] += rating
            hotel_rating_count[hotel] += 1

        for a in aspects:
            val = getattr(row, a)
            if pd.notna(val):
                hotel_aspect_sum[a][hotel] += val
                hotel_aspect_count[a][hotel] += 1

    out_df = pd.DataFrame(rows_out)

    out_df.to_csv(
        output_path,
        mode="w" if first_write else "a",
        header=first_write,
        index=False
    )
    first_write = False

    chunk_time = time.time() - chunk_start
    total_time = time.time() - start_time

    # print(f"Chunk {chunk_num} done in {chunk_time:.2f} sec")
    # print(f"Total elapsed time: {total_time/60:.2f} minutes")
    # print(f"Last date in chunk: {chunk['date'].iloc[-1]}")

    print(f"Chunk {chunk_num} done in {chunk_time:.2f} sec", flush=True)
    print(f"Total elapsed time: {total_time/60:.2f} minutes", flush=True)
    print(f"Last date in chunk: {chunk['date'].iloc[-1]}", flush=True)

print("\nFinished streaming feature generation.")
print(f"Output saved to: {output_path}")
print(f"Total rows processed: {total_rows_processed:,}")

Streaming output truncated to the last 5000 lines.
Chunk size: 50,000
Total rows processed so far: 14,650,000
Chunk 293 done in 1.92 sec
Total elapsed time: 19.20 minutes
Last date in chunk: 2014-03-01T00:00:00

--- Processing chunk 294 ---
Chunk size: 50,000
Total rows processed so far: 14,700,000
Chunk 294 done in 1.86 sec
Total elapsed time: 19.24 minutes
Last date in chunk: 2014-03-01T00:00:00

--- Processing chunk 295 ---
Chunk size: 50,000
Total rows processed so far: 14,750,000
Chunk 295 done in 1.85 sec
Total elapsed time: 19.27 minutes
Last date in chunk: 2014-03-01T00:00:00

--- Processing chunk 296 ---
Chunk size: 50,000
Total rows processed so far: 14,800,000
Chunk 296 done in 1.84 sec
Total elapsed time: 19.31 minutes
Last date in chunk: 2014-04-01T00:00:00

--- Processing chunk 297 ---
Chunk size: 50,000
Total rows processed so far: 14,850,000
Chunk 297 done in 1.94 sec
Total elapsed time: 19.41 minutes
Last date in chunk: 2014-04-01T00:00:00

--- Processing chunk 298 ---

In [ ]:
!cp "/content/hotelrec_model_ready_full_streamed.csv" "/content/drive/MyDrive/hotelrec_model_ready_full_streamed.csv" #copy from local colab storage back to drive to permanently save

In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

#model_df = pd.read_csv("hotelrec_model_ready_full_streamed.csv")
model_df = pd.read_csv("/content/drive/MyDrive/hotelrec_model_ready_full_streamed.csv")
print(model_df.shape)
print(model_df.columns)
print(model_df.dtypes)

# load processed csv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(50264364, 16)
Index(['date', 'rating', 'year', 'month', 'user_review_count_prior',
       'user_avg_rating_prior', 'hotel_review_count_prior',
       'hotel_avg_rating_prior', 'user_has_history', 'hotel_has_history',
       'hotel_avg_sleep_quality_prior', 'hotel_avg_value_prior',
       'hotel_avg_rooms_prior', 'hotel_avg_service_prior',
       'hotel_avg_cleanliness_prior', 'hotel_avg_location_prior'],
      dtype='object')
date                              object
rating                           float64
year                               int64
month                              int64
user_review_count_prior            int64
user_avg_rating_prior            float64
hotel_review_count_prior           int64
hotel_avg_rating_prior           float64
user_has_history                   int64
hotel_has_history                  int64
hotel_avg_sleep_quality_prior 

In [ ]:
model_df["date"] = pd.to_datetime(model_df["date"])
model_df = model_df.sort_values("date").reset_index(drop=True)
# convert date to datetime and sort to double check

In [ ]:
n = len(model_df)
train_end = int(0.95 * n)

train = model_df.iloc[:train_end]
test = model_df.iloc[train_end:]

print(train["date"].min(), train["date"].max())
print(test["date"].min(), test["date"].max())
print(train.shape, test.shape)
# temporal 95/5 train/test split

2001-02-01 00:00:00 2018-11-01 00:00:00
2018-11-01 00:00:00 2019-09-20 00:00:00
(47751145, 16) (2513219, 16)


In [ ]:
features = [c for c in model_df.columns if c not in ["date", "rating"]]

X_train = train[features]
y_train = train["rating"]

X_test = test[features]
y_test = test["rating"]
# build feature matrix

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=1.0,
    min_child_weight=5,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
    early_stopping_rounds=20
)

n_sample = min(100000, len(X_test))

val_sample = X_test.sample(n=n_sample, random_state=42)
y_val_sample = y_test.loc[val_sample.index]

start = time.time()

model.fit(
    X_train, y_train,
    eval_set=[(val_sample, y_val_sample)],
    verbose=10
)

fit_time = time.time() - start
print("Total fit time (minutes):", fit_time / 60)

preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print("Test RMSE:", rmse)

model.save_model("/content/drive/MyDrive/xgboost_hotelrec_full_model.json") # save trained model
np.save("/content/drive/MyDrive/xgb_preds.npy", preds) # save model outputs
np.save("/content/drive/MyDrive/y_test.npy", y_test.values) #save ground truth (real ratings)

# train xgboost on first temporal 95% of full dataset using previously obtained best parameters, test rmse from most recent 5%

[0]	validation_0-rmse:1.13714
[10]	validation_0-rmse:1.08975
[20]	validation_0-rmse:1.06190
[30]	validation_0-rmse:1.04557
[40]	validation_0-rmse:1.03592
[50]	validation_0-rmse:1.03015
[60]	validation_0-rmse:1.02660
[70]	validation_0-rmse:1.02432
[80]	validation_0-rmse:1.02287
[90]	validation_0-rmse:1.02186
[100]	validation_0-rmse:1.02124
[110]	validation_0-rmse:1.02073
[120]	validation_0-rmse:1.02027
[130]	validation_0-rmse:1.01997
[140]	validation_0-rmse:1.01967
[150]	validation_0-rmse:1.01941
[160]	validation_0-rmse:1.01920
[170]	validation_0-rmse:1.01902
[180]	validation_0-rmse:1.01889
[190]	validation_0-rmse:1.01874
[200]	validation_0-rmse:1.01865
[210]	validation_0-rmse:1.01853
[220]	validation_0-rmse:1.01842
[230]	validation_0-rmse:1.01833
[240]	validation_0-rmse:1.01823
[250]	validation_0-rmse:1.01814
[260]	validation_0-rmse:1.01807
[270]	validation_0-rmse:1.01802
[280]	validation_0-rmse:1.01793
[290]	validation_0-rmse:1.01787
[299]	validation_0-rmse:1.01782
Total fit time (min

In [ ]:
# evaluate rmse and mae
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
import numpy as np

preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)

print("Test RMSE:", rmse)
print("Test MAE:", mae)

Test RMSE: 1.023684999983211
Test MAE: 0.7457121807323462
